## CREATING A DATAFRAME.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import datetime

spark = SparkSession.builder\
        .appName("spark with hive") \
        .enableHiveSupport() \
        .getOrCreate()

data = [
    ["Product A", 1001, datetime.strptime("2023-07-20", "%Y-%m-%d"), datetime.strptime("2023-07-20 10:15:30", "%Y-%m-%d %H:%M:%S"), 29.99],
    ["Product B", 1002, datetime.strptime("2023-07-19", "%Y-%m-%d"), datetime.strptime("2023-07-19 14:20:45", "%Y-%m-%d %H:%M:%S"), 49.99],
    ["Product C", 1003, datetime.strptime("2023-07-18", "%Y-%m-%d"), datetime.strptime("2023-07-18 09:30:15", "%Y-%m-%d %H:%M:%S"), 39.99],
    ["Product D", 1004, datetime.strptime("2023-07-17", "%Y-%m-%d"), datetime.strptime("2023-07-17 16:45:00", "%Y-%m-%d %H:%M:%S"), 19.99]
]


schema = StructType([
    StructField("Product", StringType(), True),
    StructField("ID", IntegerType(), True),
    StructField("Date", DateType(), True),
    StructField("Timestamp", TimestampType(), True),
    StructField("Price", FloatType(), True)
])

df = spark.createDataFrame(data, schema)

df.printSchema()

df.show()

25/02/06 08:52:27 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


root
 |-- Product: string (nullable = true)
 |-- ID: integer (nullable = true)
 |-- Date: date (nullable = true)
 |-- Timestamp: timestamp (nullable = true)
 |-- Price: float (nullable = true)



+---------+----+----------+-------------------+-----+
|  Product|  ID|      Date|          Timestamp|Price|
+---------+----+----------+-------------------+-----+
|Product A|1001|2023-07-20|2023-07-20 10:15:30|29.99|
|Product B|1002|2023-07-19|2023-07-19 14:20:45|49.99|
|Product C|1003|2023-07-18|2023-07-18 09:30:15|39.99|
|Product D|1004|2023-07-17|2023-07-17 16:45:00|19.99|
+---------+----+----------+-------------------+-----+



## READING A FILE FROM HDFS

In [3]:
schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_item_id", IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", TimestampType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True)
])

hdfs_path = '/tmp/input_data/order_items_dataset.csv'


df = spark.read.format('csv').option('header', 'true').option('inferSchema' , 'false').schema(schema).load(hdfs_path)

df.printSchema()

df.show(5)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

In [4]:
df2 = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(hdfs_path)

df2.printSchema()

df2.show(5)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48

## REPARTIONING

In [5]:
print(df2.rdd.getNumPartitions())

2


In [6]:
df3 = df2.repartition(10)

print("DF3 partitions ::", df3.rdd.getNumPartitions())

DF3 partitions :: 10


## SELECT

In [7]:
df.select('order_id').show()

+--------------------+
|            order_id|
+--------------------+
|00010242fe8c5a6d1...|
|00018f77f2f0320c5...|
|000229ec398224ef6...|
|00024acbcdf0a6daa...|
|00042b26cf59d7ce6...|
|00048cc3ae777c65d...|
|00054e8431b9d7675...|
|000576fe39319847c...|
|0005a1a1728c9d785...|
|0005f50442cb953dc...|
|00061f2a7bc09da83...|
|00063b381e2406b52...|
|0006ec9db01a64e59...|
|0008288aa423d2a3f...|
|0008288aa423d2a3f...|
|0009792311464db53...|
|0009c9a17f916a706...|
|000aed2e25dbad2f9...|
|000c3e6612759851c...|
|000e562887b1f2006...|
+--------------------+
only showing top 20 rows



In [8]:
from pyspark.sql.functions import *

In [9]:
df.select(col('order_id'), col('product_id')).show(5)

+--------------------+--------------------+
|            order_id|          product_id|
+--------------------+--------------------+
|00010242fe8c5a6d1...|4244733e06e7ecb49...|
|00018f77f2f0320c5...|e5f2d52b802189ee6...|
|000229ec398224ef6...|c777355d18b72b67a...|
|00024acbcdf0a6daa...|7634da152a4610f15...|
|00042b26cf59d7ce6...|ac6c3623068f30de0...|
+--------------------+--------------------+
only showing top 5 rows



In [10]:
df.select(col('order_id').alias("ORDERS"), col('product_id')).show(5)

+--------------------+--------------------+
|              ORDERS|          product_id|
+--------------------+--------------------+
|00010242fe8c5a6d1...|4244733e06e7ecb49...|
|00018f77f2f0320c5...|e5f2d52b802189ee6...|
|000229ec398224ef6...|c777355d18b72b67a...|
|00024acbcdf0a6daa...|7634da152a4610f15...|
|00042b26cf59d7ce6...|ac6c3623068f30de0...|
+--------------------+--------------------+
only showing top 5 rows



## DERIVING NEW COLUMNS USING `WITH`.

In [11]:
df.show(5)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

In [12]:
df4 = df.withColumn("YEAR", year(col("shipping_limit_date"))).withColumn("month", month(col("shipping_limit_date")))
                    
df4.show(4)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+----+-----+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|YEAR|month|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+----+-----+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|2017|    9|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|2017|    5|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|2018|    1|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|2018|    8|
+--------------------+-------------+--------------------+--------------------+------------

## RENAME COLUMNS

In [13]:
df5 = df4.withColumnRenamed('shipping_limit_date', 'shipping_limit_datetime')
df5.select('order_id', 'shipping_limit_datetime').show(5)

+--------------------+-----------------------+
|            order_id|shipping_limit_datetime|
+--------------------+-----------------------+
|00010242fe8c5a6d1...|    2017-09-19 09:45:35|
|00018f77f2f0320c5...|    2017-05-03 11:05:13|
|000229ec398224ef6...|    2018-01-18 14:48:30|
|00024acbcdf0a6daa...|    2018-08-15 10:10:18|
|00042b26cf59d7ce6...|    2017-02-13 13:57:51|
+--------------------+-----------------------+
only showing top 5 rows



## FILTERING

In [14]:
df5.filter(col("order_id") == '00010242fe8c5a6d1ba2dd792cb16214').show()

+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime|price|freight_value|YEAR|month|
+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|    2017-09-19 09:45:35| 58.9|        13.29|2017|    9|
+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+



In [15]:
order_li = ['00010242fe8c5a6d1ba2dd792cb16214','00018f77f2f0320c557190d7a144bdd3']
df5.filter(col("order_id").isin(order_li)).show(5)

+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime|price|freight_value|YEAR|month|
+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|    2017-09-19 09:45:35| 58.9|        13.29|2017|    9|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|    2017-05-03 11:05:13|239.9|        19.93|2017|    5|
+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+



In [16]:
df5.filter(col("price") > 50).show()

+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+-----+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime| price|freight_value|YEAR|month|
+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+-----+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|    2017-09-19 09:45:35|  58.9|        13.29|2017|    9|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|    2017-05-03 11:05:13| 239.9|        19.93|2017|    5|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|    2018-01-18 14:48:30| 199.0|        17.87|2018|    1|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|    2017-02-13 13:57:51| 199.9|        18.14|2017|    2|
|000576fe39319847c...|            1|557d850972a7d6f79..

In [17]:
df5.filter((col("price") > 100 ) & (col("freight_value") < 25)).show(5)

+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+-----+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime| price|freight_value|YEAR|month|
+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+-----+
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|    2017-05-03 11:05:13| 239.9|        19.93|2017|    5|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|    2018-01-18 14:48:30| 199.0|        17.87|2018|    1|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|    2017-02-13 13:57:51| 199.9|        18.14|2017|    2|
|0005a1a1728c9d785...|            1|310ae3c140ff94b03...|a416b6a846a117243...|    2018-03-26 18:31:29|145.95|        11.65|2018|    3|
|0009c9a17f916a706...|            1|3f27ac8e699df3d30..

In [18]:
df5.filter("price > 100 and freight_value < 25").show(10)

+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+-----+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime| price|freight_value|YEAR|month|
+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+-----+
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|    2017-05-03 11:05:13| 239.9|        19.93|2017|    5|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|    2018-01-18 14:48:30| 199.0|        17.87|2018|    1|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|    2017-02-13 13:57:51| 199.9|        18.14|2017|    2|
|0005a1a1728c9d785...|            1|310ae3c140ff94b03...|a416b6a846a117243...|    2018-03-26 18:31:29|145.95|        11.65|2018|    3|
|0009c9a17f916a706...|            1|3f27ac8e699df3d30..

## DROP DUPLICATES

In [19]:
df5.dropDuplicates().show(5)

+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+-----+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime| price|freight_value|YEAR|month|
+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+-----+
|a6ddd2889891733e0...|            2|01084e8138d03dc69...|3092c0b297aacfb4b...|    2018-03-06 12:50:32|  44.9|        11.73|2018|    3|
|ac24c75566d21c202...|            2|4c36e30350c41feb8...|4736e9d642ef4257c...|    2017-11-30 10:52:23|139.99|        14.09|2017|   11|
|af9cf6a8b011e9fea...|            1|6fdfdddfa3c987233...|1b8b75e227c9a9c10...|    2017-01-20 23:17:31| 89.99|        14.66|2017|    1|
|afbf35931e267fd79...|            1|d42869d5edf603eb8...|af3ef48d0e13835e5...|    2018-06-06 03:30:40|  53.0|        12.81|2018|    6|
|b03c3813b79abede0...|            1|93c480c7d11c68ba0..

In [20]:
df5.dropDuplicates(['order_id', 'order_item_id']).show(5)

+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+-----+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime| price|freight_value|YEAR|month|
+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+-----+
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|    2017-05-03 11:05:13| 239.9|        19.93|2017|    5|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|    2018-01-18 14:48:30| 199.0|        17.87|2018|    1|
|00048cc3ae777c65d...|            1|ef92defde845ab845...|6426d21aca402a131...|    2017-05-23 03:55:27|  21.9|        12.69|2017|    5|
|0005a1a1728c9d785...|            1|310ae3c140ff94b03...|a416b6a846a117243...|    2018-03-26 18:31:29|145.95|        11.65|2018|    3|
|0005f50442cb953dc...|            1|4535b0e1091c278df..

## DROP A COLUMN.

In [21]:
df5.drop('month').show()  # this will create a new dataframe as dataframes are immutable.

+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime| price|freight_value|YEAR|
+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|    2017-09-19 09:45:35|  58.9|        13.29|2017|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|    2017-05-03 11:05:13| 239.9|        19.93|2017|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|    2018-01-18 14:48:30| 199.0|        17.87|2018|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|    2018-08-15 10:10:18| 12.99|        12.79|2018|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|    2017-02-13 13:5

## GET DISTINCT ROWS.

In [22]:
df5.distinct().show()

+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+-----+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime| price|freight_value|YEAR|month|
+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+-----+
|a6ddd2889891733e0...|            2|01084e8138d03dc69...|3092c0b297aacfb4b...|    2018-03-06 12:50:32|  44.9|        11.73|2018|    3|
|ac24c75566d21c202...|            2|4c36e30350c41feb8...|4736e9d642ef4257c...|    2017-11-30 10:52:23|139.99|        14.09|2017|   11|
|af9cf6a8b011e9fea...|            1|6fdfdddfa3c987233...|1b8b75e227c9a9c10...|    2017-01-20 23:17:31| 89.99|        14.66|2017|    1|
|afbf35931e267fd79...|            1|d42869d5edf603eb8...|af3ef48d0e13835e5...|    2018-06-06 03:30:40|  53.0|        12.81|2018|    6|
|b03c3813b79abede0...|            1|93c480c7d11c68ba0..

## ARRANGE DATA USING `ORDER BY`.

In [23]:
df5.orderBy(col('price').desc()).show()

+--------------------+-------------+--------------------+--------------------+-----------------------+-------+-------------+----+-----+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime|  price|freight_value|YEAR|month|
+--------------------+-------------+--------------------+--------------------+-----------------------+-------+-------------+----+-----+
|0812eb902a67711a1...|            1|489ae2aa008f02150...|e3b4998c7a498169d...|    2017-02-16 20:37:36| 6735.0|       194.31|2017|    2|
|fefacc66af859508b...|            1|69c590f7ffc7bf8db...|80ceebb4ee9b31afb...|    2018-08-02 04:05:13| 6729.0|       193.21|2018|    8|
|f5136e38d1a14a4db...|            1|1bdf5e6731585cf01...|ee27a8f15b1dded4d...|    2017-06-15 02:45:17| 6499.0|       227.66|2017|    6|
|a96610ab360d42a2e...|            1|a6492cc69376c469a...|59417c56835dd8e2e...|    2017-04-18 13:25:18| 4799.0|       151.34|2017|    4|
|199af31afc78c699f...|            1|c3ed642d5925

In [24]:
df5.orderBy(col('price').desc(), col('freight_value')).show()

+--------------------+-------------+--------------------+--------------------+-----------------------+-------+-------------+----+-----+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime|  price|freight_value|YEAR|month|
+--------------------+-------------+--------------------+--------------------+-----------------------+-------+-------------+----+-----+
|0812eb902a67711a1...|            1|489ae2aa008f02150...|e3b4998c7a498169d...|    2017-02-16 20:37:36| 6735.0|       194.31|2017|    2|
|fefacc66af859508b...|            1|69c590f7ffc7bf8db...|80ceebb4ee9b31afb...|    2018-08-02 04:05:13| 6729.0|       193.21|2018|    8|
|f5136e38d1a14a4db...|            1|1bdf5e6731585cf01...|ee27a8f15b1dded4d...|    2017-06-15 02:45:17| 6499.0|       227.66|2017|    6|
|a96610ab360d42a2e...|            1|a6492cc69376c469a...|59417c56835dd8e2e...|    2017-04-18 13:25:18| 4799.0|       151.34|2017|    4|
|199af31afc78c699f...|            1|c3ed642d5925


## GROUP BY 

In [25]:
df5.groupby('year').agg(count("*").alias("TOTAL COUNT"),
                        avg('price').alias("AVERAGE_PRICE"),
                        sum('price').alias("SUM_PRICE"),
                        min('price').alias("MIN_PRICE"),
                        max('price').alias("MAX_PRICE")).orderBy(col('year').desc()).show()

+----+-----------+------------------+-----------------+---------+---------+
|year|TOTAL COUNT|     AVERAGE_PRICE|        SUM_PRICE|MIN_PRICE|MAX_PRICE|
+----+-----------+------------------+-----------------+---------+---------+
|2020|          4|             86.49|           345.96|    69.99|    99.99|
|2018|      62511|120.08515685240229| 7506643.24000052|     0.85|   6729.0|
|2017|      49765|121.26732804178806|6034868.579999583|      1.2|   6735.0|
|2016|        370|134.55654054054082| 49785.9200000001|      6.0|   1399.0|
+----+-----------+------------------+-----------------+---------+---------+



In [26]:
df5.groupby('year', 'month').agg(count("*").alias("TOTAL COUNT"),
                        avg('price').alias("AVERAGE_PRICE"),
                        sum('price').alias("SUM_PRICE"),
                        min('price').alias("MIN_PRICE"),
                        max('price').alias("MAX_PRICE")).orderBy(col('year').desc(), col('month').asc()).show()

+----+-----+-----------+------------------+------------------+---------+---------+
|year|month|TOTAL COUNT|     AVERAGE_PRICE|         SUM_PRICE|MIN_PRICE|MAX_PRICE|
+----+-----+-----------+------------------+------------------+---------+---------+
|2020|    2|          2|             72.99|            145.98|    69.99|    75.99|
|2020|    4|          2|             99.99|            199.98|    99.99|    99.99|
|2018|    1|       7492|112.82423651895472| 845279.1800000088|     4.78|   2110.0|
|2018|    2|       7375|111.08187796610255| 819228.8500000064|     2.99|   3690.0|
|2018|    3|       8759|117.65359515926707|1030527.8400000202|      4.5|   3700.0|
|2018|    4|       7637|125.53747544847617| 958729.7000000125|     4.99|  4099.99|
|2018|    5|       8765| 123.7146092413026|1084358.5500000173|     0.85|   3930.0|
|2018|    6|       6897|126.24863563868458| 870736.8400000075|      3.5|   4590.0|
|2018|    7|       6672|121.57255395683512| 811132.0800000039|      3.0|   3089.0|
|201

## FILL MISSING VALUES.

In [27]:
df5.fillna({'price':0, 'freight_value': 0 }).filter(col('price')==0).show(5)

+--------+-------------+----------+---------+-----------------------+-----+-------------+----+-----+
|order_id|order_item_id|product_id|seller_id|shipping_limit_datetime|price|freight_value|YEAR|month|
+--------+-------------+----------+---------+-----------------------+-----+-------------+----+-----+
+--------+-------------+----------+---------+-----------------------+-----+-------------+----+-----+



## PERFORMING ACCUMULATION
`accumulator = a variable that will be global and every partition can access it.`



In [28]:
accum = spark.sparkContext.accumulator(0)

df5.foreach(lambda row: accum.add(row['price']))

print("TOTAL PRICE ACROSS ALL PARTITIONS :: ", accum)

TOTAL PRICE ACROSS ALL PARTITIONS ::  13591643.70000748


## PERFORMING CASE WHEN.

In [29]:
df5.withColumn("price_category", when(col("price") >= 100, "high")
                                    .when((col("price") < 100) & (col("price") >= 50), "medium")
                                    .otherwise("low")).show(5)

+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+--------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime|price|freight_value|YEAR|month|price_category|
+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+--------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|    2017-09-19 09:45:35| 58.9|        13.29|2017|    9|        medium|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|    2017-05-03 11:05:13|239.9|        19.93|2017|    5|          high|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|    2018-01-18 14:48:30|199.0|        17.87|2018|    1|          high|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|    2018-08-15 10:10:18|12.9

## WINDOW FUNCTION

In [30]:
from pyspark.sql.window import Window

windowspec1 = Window.partitionBy('year').orderBy(col('price').asc())
df5.withColumn('dense_rank', dense_rank().over(windowspec1)).select(col('YEAR'), col("month"),col("price"), col("dense_rank")).show()


+----+-----+-----+----------+
|YEAR|month|price|dense_rank|
+----+-----+-----+----------+
|2016|   10|  6.0|         1|
|2016|   10|  9.9|         2|
|2016|   10| 10.0|         3|
|2016|   12| 10.9|         4|
|2016|   10| 11.9|         5|
|2016|   10| 14.9|         6|
|2016|   10| 14.9|         6|
|2016|   10| 15.0|         7|
|2016|   10| 15.9|         8|
|2016|   10| 16.9|         9|
|2016|   10| 17.8|        10|
|2016|   10| 18.9|        11|
|2016|   10| 18.9|        11|
|2016|   10| 18.9|        11|
|2016|   10| 19.0|        12|
|2016|   10|19.83|        13|
|2016|   10|19.83|        13|
|2016|   10|19.83|        13|
|2016|   10| 19.9|        14|
|2016|   10| 19.9|        14|
+----+-----+-----+----------+
only showing top 20 rows



In [31]:
windowspec2 = Window.partitionBy('year', 'month').orderBy(col('shipping_limit_datetime').asc())
df5.withColumn('running_sum', sum('price').over(windowspec2)).select('year', 'month', 'shipping_limit_datetime', 'price','running_sum').show()

+----+-----+-----------------------+-----+------------------+
|year|month|shipping_limit_datetime|price|       running_sum|
+----+-----+-----------------------+-----+------------------+
|2016|    9|    2016-09-19 00:15:34| 59.5|              59.5|
|2016|    9|    2016-09-19 23:11:33|44.99|194.47000000000003|
|2016|    9|    2016-09-19 23:11:33|44.99|194.47000000000003|
|2016|    9|    2016-09-19 23:11:33|44.99|194.47000000000003|
|2016|   10|    2016-10-08 10:34:01|29.99|             29.99|
|2016|   10|    2016-10-08 10:45:33|189.0|            218.99|
|2016|   10|    2016-10-08 13:26:12|  9.9|228.89000000000001|
|2016|   10|    2016-10-08 13:46:32| 23.9|            276.69|
|2016|   10|    2016-10-08 13:46:32| 23.9|            276.69|
|2016|   10|    2016-10-08 13:47:45| 67.9|344.59000000000003|
|2016|   10|    2016-10-08 14:09:08| 89.9|            434.49|
|2016|   10|    2016-10-08 14:27:50|379.9|            814.39|
|2016|   10|    2016-10-08 14:46:49| 93.9|            908.29|
|2016|  

## reading another file from hdfs

In [32]:
hdfs_path_1 = '/tmp/input_data/sellers_dataset.csv'
sdf = spark.read.format('csv').option('header', 'true').option('inferschema', 'true').load(hdfs_path_1)

sdf.printSchema()

sdf.show(6)

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)

+--------------------+----------------------+-----------------+------------+
|           seller_id|seller_zip_code_prefix|      seller_city|seller_state|
+--------------------+----------------------+-----------------+------------+
|3442f8959a84dea7e...|                 13023|         campinas|          SP|
|d1b65fc7debc3361e...|                 13844|       mogi guacu|          SP|
|ce3ad9de960102d06...|                 20031|   rio de janeiro|          RJ|
|c0f3eea2e14555b6f...|                  4195|        sao paulo|          SP|
|51a04a8a6bdcb23de...|                 12914|braganca paulista|          SP|
|c240c4061717ac180...|                 20920|   rio de janeiro|          RJ|
+--------------------+----------------------+-----------------+------------+
only showing top 6 rows



In [33]:
sdf.count()

3095

## BROADCAST JOIN.

In [34]:
result1 = df5.join(broadcast(sdf), df5.seller_id == sdf.seller_id, 'inner').drop(sdf.seller_id)

result1.show()

+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+-----+----------------------+--------------------+------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime| price|freight_value|YEAR|month|seller_zip_code_prefix|         seller_city|seller_state|
+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+-----+----------------------+--------------------+------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|    2017-09-19 09:45:35|  58.9|        13.29|2017|    9|                 27277|       volta redonda|          SP|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|    2017-05-03 11:05:13| 239.9|        19.93|2017|    5|                  3471|           sao paulo|          SP|
|000229ec398224ef6...|            1|c777

## JOIN WITH ALIAS

In [35]:
result2 = df5.alias('od').join(sdf.alias('sellers'), col('od.seller_id') == col('sellers.seller_id'), 'inner').drop(col('sellers.seller_id'))

result2.show(5)

+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+----------------------+-------------+------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime|price|freight_value|YEAR|month|seller_zip_code_prefix|  seller_city|seller_state|
+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+----------------------+-------------+------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|    2017-09-19 09:45:35| 58.9|        13.29|2017|    9|                 27277|volta redonda|          SP|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|    2017-05-03 11:05:13|239.9|        19.93|2017|    5|                  3471|    sao paulo|          SP|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|  

## spark SQL

In [36]:
df5.createOrReplaceTempView("ORDER_ITEM")
sdf.createOrReplaceTempView("SELLERS")


joinDF2 = spark.sql("select * from ORDER_ITEM oid INNER JOIN SELLERS sid ON oid.seller_id == sid.seller_id")

In [37]:
joinDF2.show()

+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+-----+--------------------+----------------------+--------------------+------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime| price|freight_value|YEAR|month|           seller_id|seller_zip_code_prefix|         seller_city|seller_state|
+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+-----+--------------------+----------------------+--------------------+------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|    2017-09-19 09:45:35|  58.9|        13.29|2017|    9|48436dade18ac8b2b...|                 27277|       volta redonda|          SP|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|    2017-05-03 11:05:13| 239.9|        19.93|2017|    5|dd7ddc04e1b6c2

## Write data in HDFS without any partition key

In [38]:
result1.write.format('csv').option('header', 'true').option('delimiter', ',').save('/tmp/output_data/result1/')
print("Write Successfull")

Write Successfull


In [39]:
result1.write.partitionBy('year').format('csv').option('header', 'true').option('delimeter', ',').save('/tmp/output_data/result2/')
print("Write Successfull")

Write Successfull


In [41]:
result1.coalesce(1).write.format('csv').option('header', 'true').option('delimiter', ',').save('/tmp/output_data/result3/')
print("Write Successfull")

Write Successfull


## DUMPING DATA TO A HIVE TABLE USING spark.

In [44]:
spark.sql("""set hive.exec.dynamic.partition.mode=nonstrict""")

spark.sql("""USE spark_hive""")

spark.sql("""
    Create Table if not exists orders_sellers_data (
        order_id STRING,
        order_item_id INT,
        poduct_id STRING,
        price double,
        frieght_value DOUBLE,
        seller_city STRING
        )
    PARTITIONED BY (year INT) 
""")


result1.select('order_id',
              'order_item_id',
              'product_id',
              'price',
              'freight_value',
              'seller_city',
              'year').write.mode("append").insertInto("orders_sellers_data")

25/02/06 09:24:00 WARN SetCommand: 'SET hive.exec.dynamic.partition.mode=nonstrict' might not work, since Spark doesn't support changing the Hive config dynamically. Please pass the Hive-specific config by adding the prefix spark.hadoop (e.g. spark.hadoop.hive.exec.dynamic.partition.mode) when starting a Spark application. For details, see the link: https://spark.apache.org/docs/latest/configuration.html#dynamically-loading-spark-properties.
25/02/06 09:24:00 WARN ResolveSessionCatalog: A Hive serde table will be created as there is no table provider specified. You can set spark.sql.legacy.createHiveTableByDefault to false so that native data source table will be created instead.
25/02/06 09:24:01 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
